# asbflow Quickstart Notebook

This notebook extends the [Quick Start section in README](../README.md#quick-start) with a richer live demo flow.

It is inspired by the `test_asbflow.py` integration notebook, but adapted to the current `asbflow` API.

## What This Covers

- Publisher usage with parsed and raw payloads
- Per-call static and dynamic message metadata (ASBMessageConfig / ASBDynamicMessageConfig)
- Consumer consume(...) vs ead(...) behavior
- Parse failure policies (ABANDON and DEAD_LETTER)
- DLQ manager operations: ead, edrive, purge
- Execution mode examples (sequential, 	hread_pool, sync)


In [1]:
from __future__ import annotations

import os
from datetime import datetime, timezone
from typing import Any
from uuid import uuid4

from pydantic import BaseModel

from asbflow import (
    ASBConnectionConfig,
    ASBConsumer,
    ASBConsumerConfig,
    ASBDLQManager,
    ASBDynamicMessageConfig,
    ASBMessageConfig,
    ASBPublisher,
    ASBPublisherConfig,
    MessageFieldMapping,
    ParseFailurePolicy,
    PydanticModelParser,
)


In [2]:
class AlertPayload(BaseModel):
    id: str
    severity: str
    source: str
    timestamp: str


def build_valid_payload(prefix: str, idx: int) -> dict[str, Any]:
    now = datetime.now(timezone.utc).isoformat()
    return {
        "id": f"{prefix}-{idx}",
        "severity": "high" if idx % 2 == 0 else "medium",
        "source": "quickstart",
        "timestamp": now,
    }


def build_malformed_payload(prefix: str, idx: int) -> dict[str, Any]:
    payload = build_valid_payload(prefix, idx)
    payload.pop("severity")
    return payload

In [4]:
connection_string = os.getenv("ASB_CONNECTION_STRING", "<connection-string>")
topic_name = os.getenv("ASB_TOPIC_NAME", "topic-name")
subscription_name = os.getenv("ASB_SUBSCRIPTION_NAME", "subscription-name")
run_id = uuid4().hex[:8]

connection = ASBConnectionConfig(connection_string=connection_string)
publisher_config = ASBPublisherConfig(topic_name=topic_name)
message_config = ASBMessageConfig(subject="quickstart", content_type="application/json")

consumer_abandon_config = ASBConsumerConfig(
    topic_name=topic_name,
    subscription_name=subscription_name,
    parse_failure_policy=ParseFailurePolicy.ABANDON,
    max_wait_time=10.0,
)
consumer_deadletter_config = ASBConsumerConfig(
    topic_name=topic_name,
    subscription_name=subscription_name,
    parse_failure_policy=ParseFailurePolicy.DEAD_LETTER,
    max_wait_time=10.0,
)

parser = PydanticModelParser(AlertPayload)

publisher = ASBPublisher(
    connection=connection,
    publisher=publisher_config,
    message=message_config,
    parser=parser,
    execution_mode="sequential",
)
publisher_thread_pool = ASBPublisher(
    connection=connection,
    publisher=publisher_config,
    message=message_config,
    parser=parser,
    execution_mode="thread_pool",
)

consumer_abandon = ASBConsumer(
    connection=connection,
    consumer=consumer_abandon_config,
    parser=parser,
    raise_on_error=False,
)
consumer_deadletter = ASBConsumer(
    connection=connection,
    consumer=consumer_deadletter_config,
    parser=parser,
    raise_on_error=False,
)

dlq_manager = ASBDLQManager(
    consumer=consumer_deadletter,
    publisher=publisher,
    raise_on_error=False,
)

print(f"topic={topic_name}, subscription={subscription_name}, run_id={run_id}")

topic=topic-name, subscription=subscription-name, run_id=1ac4ab82


## Live Guard

Live operations are disabled by default. Set `RUN_LIVE = True` only when your environment variables are configured.

In [5]:
RUN_LIVE = False


def require_live() -> bool:
    if not RUN_LIVE:
        print("RUN_LIVE=False: live Service Bus operations are skipped.")
        return False

    if not connection_string:
        raise ValueError("ASB_CONNECTION_STRING is required when RUN_LIVE=True")

    return True


require_live()

RUN_LIVE=False: live Service Bus operations are skipped.


False

In [6]:
if require_live():
    pre_drain = consumer_abandon.consume_all(
        max_message_count=100,
        parse=False,
        raise_on_error=False,
    )
    pre_purge = dlq_manager.purge(max_message_count=1_000, raise_on_error=False)

    print("Pre-clean completed")
    print("  drained active messages:", len(pre_drain.successes))
    print("  purged DLQ messages:", pre_purge.purged)

RUN_LIVE=False: live Service Bus operations are skipped.


## Publish Valid + Malformed Payloads (with MessageConfig overrides)


In [7]:
valid_payloads = [build_valid_payload(f"valid-{run_id}", i) for i in range(3)]
malformed_payloads = [build_malformed_payload(f"bad-{run_id}", i) for i in range(2)]

static_message_override = ASBMessageConfig(
    subject=f"quickstart-static-{run_id}",
    content_type="application/json",
)

dynamic_message_config = ASBDynamicMessageConfig(
    message_id=MessageFieldMapping(
        lambda payload: payload.get("id") if isinstance(payload, dict) else None
    ),
    subject=MessageFieldMapping(
        lambda payload: f"quickstart-dynamic-{payload.get('type', 'unknown')}"
        if isinstance(payload, dict)
        else None
    ),
    correlation_id=MessageFieldMapping(
        lambda payload: run_id if isinstance(payload, dict) else None
    ),
)

preview = dynamic_message_config.to_message_config(valid_payloads[0])
print("Dynamic config preview for first payload:", preview.to_message_kwargs())

if require_live():
    publisher.publish(
        valid_payloads[0],
        parse=False,
        message=static_message_override,
    )
    publisher.publish_batch(
        valid_payloads[1:],
        parse=False,
        chunk_size=2,
        message=dynamic_message_config,
    )
    publisher_thread_pool.publish_batch(malformed_payloads, parse=False, chunk_size=2)

    print("Published:")
    print("  valid with static override:", 1)
    print("  valid with dynamic override:", len(valid_payloads) - 1)
    print("  malformed (raw):", len(malformed_payloads))


RUN_LIVE=False: live Service Bus operations are skipped.


## Consume vs Read

In [8]:
if require_live():
    parsed_result = consumer_abandon.consume(
        max_message_count=20,
        parse=True,
        raise_on_error=False,
    )

    parsed_ids = [
        payload.id
        for payload in parsed_result.successes
        if payload.id.startswith(f"valid-{run_id}")
    ]
    print("consume(parse=True) parsed ids:", parsed_ids)
    print("consume(parse=True) failures:", len(parsed_result.failures))

    raw_read_result = consumer_abandon.read(
        max_message_count=20,
        parse=False,
        raise_on_error=False,
    )
    raw_ids = [
        payload.get("id")
        for payload in raw_read_result.successes
        if str(payload.get("id", "")).startswith(f"bad-{run_id}")
    ]
    print("read(parse=False) malformed ids still visible (not settled):", raw_ids)

RUN_LIVE=False: live Service Bus operations are skipped.


## Force DLQ + DLQ Manager (`read`, `redrive`, `purge`)

In [9]:
if require_live():
    dlq_payloads = [build_malformed_payload(f"dlq-{run_id}", i) for i in range(2)]
    publisher.publish_batch(dlq_payloads, parse=False, chunk_size=2)

    force_dlq_result = consumer_deadletter.consume_all(
        max_message_count=50,
        parse=True,
        raise_on_error=False,
    )
    print("Forced-to-DLQ failures:", len(force_dlq_result.failures))

    dlq_raw = dlq_manager.read(max_message_count=50, parse=False, raise_on_error=False)
    dlq_raw_ids = [
        payload.get("id")
        for payload in dlq_raw.successes
        if str(payload.get("id", "")).startswith(f"dlq-{run_id}")
    ]
    print("DLQ raw ids:", dlq_raw_ids)

    redrive_result = dlq_manager.redrive(
        max_message_count=50,
        parse=False,
        raise_on_error=False,
    )
    print("Redrive result:", redrive_result)

    purge_result = dlq_manager.purge(max_message_count=1_000, raise_on_error=False)
    print("Final DLQ purge result:", purge_result)

RUN_LIVE=False: live Service Bus operations are skipped.


## Async Execution Mode Example

In [10]:
async_consumer = ASBConsumer(
    connection=connection,
    consumer=consumer_abandon_config,
    parser=parser,
    execution_mode="async",
    raise_on_error=False,
)


async def demo_async_consume() -> None:
    if not require_live():
        return

    result = await async_consumer.aconsume(
        max_message_count=10,
        parse=False,
        raise_on_error=False,
    )
    print("Async consume successes:", len(result.successes))
    print("Async consume failures:", len(result.failures))


print("Run asyncio.run(demo_async_consume()) after setting RUN_LIVE=True if you want to test async mode.")

Run asyncio.run(demo_async_consume()) after setting RUN_LIVE=True if you want to test async mode.
